# SRGL Synthetic Data Generation

This notebook demonstrates how to generate synthetic clinical cases for SRGL evaluation.

## Overview
- Generate realistic synthetic patient cases based on clinical distributions
- Create balanced datasets across risk tiers (R1-R5)
- Save datasets for use in subsequent analysis

## Requirements
Make sure you have installed all dependencies:
```bash
pip install -r requirements.txt
```

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_generator import SyntheticDataGenerator, RiskTier

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-paper')
sns.set_palette('colorblind')
%matplotlib inline

## 1. Initialize Data Generator

Create the synthetic data generator with a fixed random seed for reproducibility.

In [ ]:
# Initialize generator
generator = SyntheticDataGenerator(random_seed=42)

print("Synthetic Data Generator initialized")
print(f"Random seed: 42")
print(f"Available diagnoses: {len(generator.diagnosis_templates)}")

## 2. Generate Training Dataset

Generate 600 cases with realistic risk distribution:
- R1 (Critical): 30%
- R2 (Urgent): 20%
- R3 (Standard): 25%
- R4 (Low): 15%
- R5 (Minimal): 10%

In [ ]:
# Generate training dataset
train_cases = generator.generate_dataset(
    n_cases=600,
    risk_distribution=[0.30, 0.20, 0.25, 0.15, 0.10]
)

print(f"Generated {len(train_cases)} training cases")

# Convert to DataFrame for analysis
train_df = pd.DataFrame([
    {
        'case_id': case.case_id,
        'ground_truth_tier': case.ground_truth_tier.value,
        'diagnosis': case.diagnosis,
        'age': case.patient_features['age'],
        'sex': case.patient_features['sex'],
        'n_symptoms': len(case.clinical_presentation.symptoms),
        'n_risk_factors': len(case.clinical_presentation.risk_factors),
        'has_red_flags': len(case.clinical_presentation.red_flags) > 0
    }
    for case in train_cases
])

train_df.head()

## 3. Analyze Dataset Distribution

In [ ]:
# Risk tier distribution
print("\nRisk Tier Distribution:")
print(train_df['ground_truth_tier'].value_counts().sort_index())
print(f"\nPercentages:")
print((train_df['ground_truth_tier'].value_counts(normalize=True) * 100).sort_index().round(1))

In [ ]:
# Visualize risk distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
risk_counts = train_df['ground_truth_tier'].value_counts().sort_index()
axes[0].bar(range(1, 6), risk_counts.values, color=sns.color_palette('colorblind', 5))
axes[0].set_xlabel('Risk Tier', fontsize=12)
axes[0].set_ylabel('Number of Cases', fontsize=12)
axes[0].set_title('Risk Tier Distribution', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(1, 6))
axes[0].set_xticklabels(['R1\n(Critical)', 'R2\n(Urgent)', 'R3\n(Standard)', 'R4\n(Low)', 'R5\n(Minimal)'])
axes[0].grid(axis='y', alpha=0.3)

# Add counts on bars
for i, v in enumerate(risk_counts.values):
    axes[0].text(i+1, v + 3, str(v), ha='center', va='bottom', fontweight='bold')

# Diagnosis distribution
top_diagnoses = train_df['diagnosis'].value_counts().head(10)
axes[1].barh(range(len(top_diagnoses)), top_diagnoses.values, color=sns.color_palette('colorblind', 1))
axes[1].set_yticks(range(len(top_diagnoses)))
axes[1].set_yticklabels(top_diagnoses.index)
axes[1].set_xlabel('Number of Cases', fontsize=12)
axes[1].set_title('Top 10 Diagnoses', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/dataset_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved: figures/dataset_distribution.png")

## 4. Analyze Patient Demographics

In [ ]:
# Age statistics by risk tier
print("\nAge Statistics by Risk Tier:")
age_stats = train_df.groupby('ground_truth_tier')['age'].describe()
print(age_stats.round(1))

# Sex distribution
print("\nSex Distribution:")
print(train_df['sex'].value_counts())
print(f"\nPercentage: {(train_df['sex'].value_counts(normalize=True) * 100).round(1)}")

In [ ]:
# Visualize demographics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution by risk tier
for tier in sorted(train_df['ground_truth_tier'].unique()):
    data = train_df[train_df['ground_truth_tier'] == tier]['age']
    axes[0].hist(data, alpha=0.5, label=f'R{tier}', bins=20)

axes[0].set_xlabel('Age (years)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Age Distribution by Risk Tier', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot: Age by risk tier
train_df.boxplot(column='age', by='ground_truth_tier', ax=axes[1])
axes[1].set_xlabel('Risk Tier', fontsize=12)
axes[1].set_ylabel('Age (years)', fontsize=12)
axes[1].set_title('Age Distribution by Risk Tier', fontsize=14, fontweight='bold')
axes[1].set_xticklabels(['R1', 'R2', 'R3', 'R4', 'R5'])
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.savefig('../figures/demographics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved: figures/demographics.png")

## 5. Analyze Clinical Features

In [ ]:
# Clinical feature statistics
print("\nClinical Features by Risk Tier:")
clinical_stats = train_df.groupby('ground_truth_tier')[['n_symptoms', 'n_risk_factors']].mean()
print(clinical_stats.round(2))

# Red flag presence
print("\nRed Flag Presence by Risk Tier:")
red_flag_stats = train_df.groupby('ground_truth_tier')['has_red_flags'].mean() * 100
print(red_flag_stats.round(1))

In [ ]:
# Visualize clinical features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Number of symptoms
train_df.boxplot(column='n_symptoms', by='ground_truth_tier', ax=axes[0])
axes[0].set_xlabel('Risk Tier', fontsize=12)
axes[0].set_ylabel('Number of Symptoms', fontsize=12)
axes[0].set_title('Symptoms by Risk Tier', fontsize=14, fontweight='bold')
axes[0].set_xticklabels(['R1', 'R2', 'R3', 'R4', 'R5'])

# Number of risk factors
train_df.boxplot(column='n_risk_factors', by='ground_truth_tier', ax=axes[1])
axes[1].set_xlabel('Risk Tier', fontsize=12)
axes[1].set_ylabel('Number of Risk Factors', fontsize=12)
axes[1].set_title('Risk Factors by Risk Tier', fontsize=14, fontweight='bold')
axes[1].set_xticklabels(['R1', 'R2', 'R3', 'R4', 'R5'])

# Red flag presence
red_flag_data = train_df.groupby('ground_truth_tier')['has_red_flags'].mean() * 100
axes[2].bar(range(1, 6), red_flag_data.values, color=sns.color_palette('colorblind', 5))
axes[2].set_xlabel('Risk Tier', fontsize=12)
axes[2].set_ylabel('Percentage with Red Flags (%)', fontsize=12)
axes[2].set_title('Red Flag Presence by Risk Tier', fontsize=14, fontweight='bold')
axes[2].set_xticks(range(1, 6))
axes[2].set_xticklabels(['R1', 'R2', 'R3', 'R4', 'R5'])
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('')  # Remove default title
plt.tight_layout()
plt.savefig('../figures/clinical_features.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved: figures/clinical_features.png")

## 6. Generate Test Dataset

Generate a separate test set for evaluation (200 cases).

In [ ]:
# Generate test dataset with different seed
test_generator = SyntheticDataGenerator(random_seed=2024)
test_cases = test_generator.generate_dataset(
    n_cases=200,
    risk_distribution=[0.30, 0.20, 0.25, 0.15, 0.10]
)

print(f"Generated {len(test_cases)} test cases")

# Convert to DataFrame
test_df = pd.DataFrame([
    {
        'case_id': case.case_id,
        'ground_truth_tier': case.ground_truth_tier.value,
        'diagnosis': case.diagnosis
    }
    for case in test_cases
])

print("\nTest Set Risk Distribution:")
print(test_df['ground_truth_tier'].value_counts().sort_index())

## 7. Save Datasets

In [ ]:
# Create data directory if it doesn't exist
os.makedirs('../data', exist_ok=True)

# Save training dataset
train_output = '../data/synthetic_train.json'
generator.save_dataset(train_cases, train_output)
print(f"Training dataset saved: {train_output}")

# Save test dataset
test_output = '../data/synthetic_test.json'
test_generator.save_dataset(test_cases, test_output)
print(f"Test dataset saved: {test_output}")

## 8. Summary Statistics

In [ ]:
print("="*60)
print("DATASET GENERATION SUMMARY")
print("="*60)
print(f"\nTraining Set: {len(train_cases)} cases")
print(f"Test Set: {len(test_cases)} cases")
print(f"Total: {len(train_cases) + len(test_cases)} cases")
print(f"\nRandom Seeds:")
print(f"  Training: 42")
print(f"  Test: 2024")
print(f"\nOutput Files:")
print(f"  {train_output}")
print(f"  {test_output}")
print(f"\nFigures Generated:")
print(f"  figures/dataset_distribution.png")
print(f"  figures/demographics.png")
print(f"  figures/clinical_features.png")
print("\n" + "="*60)
print("Data generation complete! Proceed to notebook 02 for evaluation.")
print("="*60)